# Putah Creek Meta-Ensemble Runner

This notebook builds a reusable soft-blended meta-ensemble from the five frozen final submissions: Eamon, Brian, Felix, Minho, and Hannah.

The base expert adapter reads each expert's exact final portfolio outputs from their frozen rerun artifacts. Those artifacts were produced by the final notebooks using each person's saved models and feature pipeline. The ensemble layer then trains a strict walk-forward Ridge router without using future target rows.


## 0. Setup

Set `ENSEMBLE_DATASETS` to any TOML dataset that has matching expert output artifacts. For the current final regime suite, the expected artifacts already exist under `runs/*four_regime_proxy_backtests/`.


In [ ]:
from __future__ import annotations

import json
import math
import os
import socket
import sys
import tempfile
import warnings
from dataclasses import dataclass
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlparse
from urllib.request import Request, urlopen

warnings.filterwarnings("ignore")

import mlflow
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Robust repo-root discovery for local notebooks, VS Code, Jupyter, and Colab-style runs.
candidates = [Path.cwd(), *Path.cwd().parents]
repo_root = next((p for p in candidates if (p / "pyproject.toml").exists() and (p / "src" / "portfolio_toolkit").exists()), None)
if repo_root is None:
    repo_root = Path("/Users/adamthorne/Desktop/Portfolio/Portfolio-Optimizer")
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from portfolio_toolkit import (
    PortfolioWeights,
    backtest_weights,
    get_dataset_spec,
    init_mlflow,
    load_prices,
    log_backtest,
    log_portfolio,
    log_predictions,
    start_run,
    validate_weights_frame,
    write_backtest_artifacts,
)

print("repo_root:", repo_root)
print("python:", sys.executable)


## 1. Ensemble Configuration

`EXPERT_OUTPUT_MODE='artifact_cache'` uses the frozen outputs from the exact final model reruns. If a new dataset is added later, run each expert's final notebook on that dataset first so the same path convention exists, then add the dataset name here.


In [ ]:
FAST_SMOKE = os.environ.get("FAST_SMOKE", "0").strip() in {"1", "true", "True", "yes"}
LOG_TO_MLFLOW = os.environ.get("PUTAH_ENSEMBLE_LOG_TO_MLFLOW", "0").strip() in {"1", "true", "True", "yes"}
MLFLOW_REQUIRED = os.environ.get("PUTAH_ENSEMBLE_MLFLOW_REQUIRED", "0").strip() in {"1", "true", "True", "yes"}

ENSEMBLE_DATASETS = [
    "regime_modern_tech_gain_2022_2026",
    "regime_financial_crisis_loss_2005_2010",
    "regime_nineties_volatility_1995_1999",
    "regime_oil_pre2014_energy_2010_2013",
]
if FAST_SMOKE:
    ENSEMBLE_DATASETS = ["regime_modern_tech_gain_2022_2026"]

ENSEMBLE_NAME = "putah_creek_trained_meta_ensemble"
OUTPUT_ROOT = repo_root / "runs" / "putah_creek_meta_ensemble"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

META_TARGET_HORIZON = 5
# 504 trading days is roughly 101 weekly rebalance observations.
META_MIN_TRAIN_DATES = int(math.ceil(504 / 5)) if not FAST_SMOKE else 24
META_RETRAIN_EVERY_N_REBALANCES = 21 if not FAST_SMOKE else 4
META_ALPHA = 10.0
REBALANCE_FREQUENCY = "weekly_first_trading_day"
MAX_WEIGHT_CAP = 0.10
SMOOTH_PREVIOUS_WEIGHT = 0.35
SMOOTH_NEW_WEIGHT = 0.65

EXPERTS = ["eamon", "brian", "felix", "minho", "hannah"]
EXPERT_DISPLAY_NAMES = {
    "eamon": "Eamon XGBoost MinVar Tilt",
    "brian": "Brian LightGBM v6.1",
    "felix": "Felix CatBoost Run13",
    "minho": "Minho RankingConfidence LSTM v4",
    "hannah": "Hannah Multi-Regime Vol Router",
}

EXPERT_WEIGHT_PATH_TEMPLATES = {
    "eamon": "runs/eamon_xgboost_final_four_regime_proxy_backtests/{dataset}/tilted/weights.parquet",
    "brian": "runs/brian_lgbm_v61_four_regime_proxy_backtests/{dataset}/weights.parquet",
    "felix": "runs/felix_run13_four_regime_proxy_backtests/{dataset}/weights.parquet",
    "minho": "runs/minho_v4_dynamic_risk_four_regime_proxy_backtests/{dataset}/weights.parquet",
    "hannah": "runs/supervised_lstm_autoencoder_multiregime_router_shared_set_2/four_regime_proxy_backtests/{dataset}/weights.parquet",
}
EXPERT_PREDICTION_PATH_TEMPLATES = {
    "eamon": "runs/eamon_xgboost_final_four_regime_proxy_backtests/{dataset}/risk_diagnostics.parquet",
    "brian": "runs/brian_lgbm_v61_four_regime_proxy_backtests/{dataset}/predictions.parquet",
    "felix": "runs/felix_run13_four_regime_proxy_backtests/{dataset}/predictions.parquet",
    "minho": "runs/minho_v4_dynamic_risk_four_regime_proxy_backtests/{dataset}/predictions.parquet",
    "hannah": "runs/supervised_lstm_autoencoder_multiregime_router_shared_set_2/four_regime_proxy_backtests/{dataset}/predictions.parquet",
}
EXPERT_SOURCE_NOTEBOOKS = {
    "eamon": "MODELS/Eamon/eamon_xgboost_final.ipynb",
    "brian": "MODELS/Brian/brian_lgbm_v61.ipynb",
    "felix": "MODELS/Felix/catboost_full_model_run13_remake.ipynb",
    "minho": "MODELS/Minho/baseline_lstm_v4_dynamic_risk.ipynb",
    "hannah": "MODELS/Hannah/supervised_lstm_autoencoder_multiregime_router.ipynb",
}

print("Datasets:", ENSEMBLE_DATASETS)
print("MLflow logging:", LOG_TO_MLFLOW)
print("FAST_SMOKE:", FAST_SMOKE)


## 2. Expert Artifact Adapters

Each adapter returns the expert's final tradable preference as weights, then standardizes those weights into long-form expert signals. A positive portfolio weight is treated as active preference. Missing expert/date/ticker coverage is represented separately with `expert_available = 0`.


In [ ]:
@dataclass(frozen=True)
class ExpertRun:
    expert: str
    dataset_name: str
    weights: pd.DataFrame
    predictions: pd.DataFrame | None
    expert_signal: pd.DataFrame
    metadata: dict[str, object]


def _read_weight_frame(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    frame = pd.read_parquet(path)
    if "date" in frame.columns:
        frame = frame.set_index("date")
    frame.index = pd.to_datetime(frame.index, utc=True).tz_localize(None)
    frame.index.name = "date"
    frame.columns = [str(col).upper() for col in frame.columns]
    frame = frame.sort_index().replace([np.inf, -np.inf], np.nan).fillna(0.0)
    frame = frame.loc[:, ~frame.columns.duplicated()]
    return frame.astype(float)


def _read_optional_prediction_frame(path: Path) -> pd.DataFrame | None:
    if not path.exists():
        return None
    frame = pd.read_parquet(path)
    if "date" in frame.columns:
        frame["date"] = pd.to_datetime(frame["date"], utc=True).dt.tz_localize(None)
    if "ticker" in frame.columns:
        frame["ticker"] = frame["ticker"].astype(str).str.upper()
    return frame


def _select_weekly_first_trading_dates(dates: pd.Series | pd.DatetimeIndex) -> pd.DatetimeIndex:
    idx = pd.DatetimeIndex(pd.to_datetime(pd.Series(dates).dropna().unique())).sort_values()
    if idx.empty:
        return idx
    frame = pd.DataFrame(index=idx)
    frame["week"] = frame.index.to_period("W-FRI")
    return pd.DatetimeIndex(frame.groupby("week", sort=True).head(1).index)


def _align_weights_to_dates(weights: pd.DataFrame, target_dates: pd.DatetimeIndex, tickers: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    target_dates = pd.DatetimeIndex(pd.to_datetime(target_dates)).sort_values()
    tickers = [ticker.upper() for ticker in tickers]
    available = pd.DataFrame(0.0, index=target_dates, columns=tickers)
    if weights.empty or target_dates.empty:
        empty = pd.DataFrame(0.0, index=target_dates, columns=tickers)
        empty.index.name = "date"
        available.index.name = "date"
        return empty, available
    working = weights.copy().reindex(columns=tickers, fill_value=0.0).sort_index()
    aligned = working.reindex(target_dates, method="ffill").fillna(0.0)
    aligned.index.name = "date"
    first_date = working.index.min()
    available.loc[target_dates >= first_date, :] = 1.0
    available.index.name = "date"
    return aligned, available


def _rank_normalized_weight_scores(weights: pd.DataFrame, availability: pd.DataFrame) -> pd.DataFrame:
    scores = pd.DataFrame(0.0, index=weights.index, columns=weights.columns)
    for date_value, row in weights.iterrows():
        row = row.astype(float).replace([np.inf, -np.inf], 0.0).fillna(0.0)
        if availability.loc[date_value].sum() <= 0:
            continue
        ranks = row.rank(method="average", pct=True)
        scores.loc[date_value] = ranks - 0.5
    scores = scores.where(availability.astype(bool), 0.0)
    scores.index.name = "date"
    return scores


def _weights_to_expert_signal(expert: str, weights: pd.DataFrame, target_dates: pd.DatetimeIndex, tickers: list[str]) -> pd.DataFrame:
    aligned, availability = _align_weights_to_dates(weights, target_dates, tickers)
    scores = _rank_normalized_weight_scores(aligned, availability)
    signal = (
        scores.stack(dropna=False)
        .rename("expert_score")
        .reset_index()
        .rename(columns={"level_1": "ticker"})
    )
    weight_long = aligned.stack(dropna=False).rename("expert_weight").reset_index().rename(columns={"level_1": "ticker"})
    avail_long = availability.stack(dropna=False).rename("expert_available").reset_index().rename(columns={"level_1": "ticker"})
    signal = signal.merge(weight_long, on=["date", "ticker"], how="left").merge(avail_long, on=["date", "ticker"], how="left")
    signal["expert"] = expert
    signal["expert_active"] = (signal["expert_weight"] > 1e-12).astype(float)
    signal["ticker"] = signal["ticker"].astype(str).str.upper()
    return signal[["date", "ticker", "expert", "expert_score", "expert_weight", "expert_active", "expert_available"]]


def run_expert_on_dataset(expert_name: str, dataset_name: str, target_dates: pd.DatetimeIndex, tickers: list[str]) -> ExpertRun:
    expert = expert_name.lower()
    if expert not in EXPERT_WEIGHT_PATH_TEMPLATES:
        raise KeyError(f"Unknown expert: {expert_name}")
    weight_path = repo_root / EXPERT_WEIGHT_PATH_TEMPLATES[expert].format(dataset=dataset_name)
    prediction_path = repo_root / EXPERT_PREDICTION_PATH_TEMPLATES[expert].format(dataset=dataset_name)
    weights = _read_weight_frame(weight_path)
    predictions = _read_optional_prediction_frame(prediction_path)
    signal = _weights_to_expert_signal(expert, weights, target_dates, tickers)
    metadata = {
        "expert": expert,
        "display_name": EXPERT_DISPLAY_NAMES[expert],
        "dataset_name": dataset_name,
        "weight_path": str(weight_path.relative_to(repo_root)),
        "prediction_path": str(prediction_path.relative_to(repo_root)) if prediction_path.exists() else None,
        "source_notebook": EXPERT_SOURCE_NOTEBOOKS[expert],
        "weight_rows": int(len(weights)),
        "weight_columns": int(len(weights.columns)),
    }
    return ExpertRun(expert=expert, dataset_name=dataset_name, weights=weights, predictions=predictions, expert_signal=signal, metadata=metadata)


## 3. Targets, Regime Features, And Meta Training Frame

The meta target is 5-trading-day forward alpha versus the dataset benchmark, converted to a date-wise cross-sectional rank in `[-0.5, 0.5]`.


In [ ]:
def _pivot_adj_close(prices: pd.DataFrame) -> pd.DataFrame:
    frame = prices.copy()
    frame["date"] = pd.to_datetime(frame["date"], utc=True).dt.tz_localize(None)
    frame["ticker"] = frame["ticker"].astype(str).str.upper()
    return frame.pivot(index="date", columns="ticker", values="adj_close").sort_index()


def _forward_alpha_rank_targets(price_wide: pd.DataFrame, tickers: list[str], benchmark: str, horizon: int) -> pd.DataFrame:
    tickers = [ticker.upper() for ticker in tickers if ticker.upper() in price_wide.columns]
    benchmark = benchmark.upper()
    if benchmark not in price_wide.columns:
        raise ValueError(f"Benchmark {benchmark} missing from price frame")
    forward_returns = price_wide[tickers].shift(-horizon) / price_wide[tickers] - 1.0
    benchmark_forward = price_wide[benchmark].shift(-horizon) / price_wide[benchmark] - 1.0
    alpha = forward_returns.sub(benchmark_forward, axis=0)
    target_rank = alpha.rank(axis=1, method="average", pct=True) - 0.5

    rows = []
    date_index = price_wide.index
    for date_value in target_rank.index:
        pos = date_index.searchsorted(date_value)
        target_pos = pos + horizon
        target_available_date = date_index[target_pos] if target_pos < len(date_index) else pd.NaT
        frame = pd.DataFrame({
            "date": date_value,
            "ticker": target_rank.columns,
            "forward_alpha_5d": alpha.loc[date_value].values,
            "target_rank_alpha_5d": target_rank.loc[date_value].values,
            "target_available_date": target_available_date,
        })
        rows.append(frame)
    return pd.concat(rows, ignore_index=True)


def _max_drawdown_window(series: pd.Series, window: int) -> pd.Series:
    rolling_max = series.rolling(window, min_periods=max(5, window // 5)).max()
    return series / rolling_max - 1.0


def _build_regime_features(price_wide: pd.DataFrame, tickers: list[str], benchmark: str) -> pd.DataFrame:
    benchmark = benchmark.upper()
    tickers = [ticker.upper() for ticker in tickers if ticker.upper() in price_wide.columns]
    returns = price_wide[tickers].pct_change(fill_method=None)
    bench_return = price_wide[benchmark].pct_change(fill_method=None)
    bench_price = price_wide[benchmark]

    features = pd.DataFrame(index=price_wide.index)
    features["benchmark_return_1d"] = bench_return
    features["benchmark_momentum_20d"] = bench_price.pct_change(20, fill_method=None)
    features["benchmark_momentum_60d"] = bench_price.pct_change(60, fill_method=None)
    features["benchmark_vol_20d_ann"] = bench_return.rolling(20, min_periods=10).std() * math.sqrt(252)
    features["benchmark_vol_60d_ann"] = bench_return.rolling(60, min_periods=20).std() * math.sqrt(252)
    features["benchmark_drawdown_60d"] = _max_drawdown_window(bench_price, 60)
    features["market_breadth_20d"] = (returns.rolling(20, min_periods=10).mean() > 0).mean(axis=1)
    features["cross_sectional_dispersion_20d"] = returns.rolling(20, min_periods=10).std().mean(axis=1)
    features["avg_universe_drawdown_60d"] = price_wide[tickers].apply(_max_drawdown_window, window=60).mean(axis=1)
    features["pct_above_sma50"] = (price_wide[tickers] > price_wide[tickers].rolling(50, min_periods=20).mean()).mean(axis=1)
    features = features.replace([np.inf, -np.inf], np.nan).ffill().fillna(0.0)
    features.index.name = "date"
    return features.reset_index()


def _build_expert_wide_features(expert_signals: pd.DataFrame) -> pd.DataFrame:
    pieces = []
    for value_col in ["expert_score", "expert_weight", "expert_active", "expert_available"]:
        wide = expert_signals.pivot_table(index=["date", "ticker"], columns="expert", values=value_col, aggfunc="last")
        wide = wide.reindex(columns=EXPERTS).fillna(0.0)
        wide.columns = [f"{expert}_{value_col}" for expert in wide.columns]
        pieces.append(wide)
    out = pd.concat(pieces, axis=1).reset_index()
    return out


def _build_meta_frame(dataset_name: str, expert_signals: pd.DataFrame, prices: pd.DataFrame, spec) -> tuple[pd.DataFrame, list[str], pd.DataFrame]:
    benchmark = spec.benchmark_ticker.upper()
    tradable = [ticker.upper() for ticker in spec.tickers if ticker.upper() != benchmark]
    price_wide = _pivot_adj_close(prices)
    targets = _forward_alpha_rank_targets(price_wide, tradable, benchmark, META_TARGET_HORIZON)
    regime = _build_regime_features(price_wide, tradable, benchmark)
    expert_wide = _build_expert_wide_features(expert_signals)
    frame = expert_wide.merge(regime, on="date", how="left").merge(targets, on=["date", "ticker"], how="left")

    base_feature_cols = []
    for expert in EXPERTS:
        base_feature_cols.extend([
            f"{expert}_expert_score",
            f"{expert}_expert_weight",
            f"{expert}_expert_active",
            f"{expert}_expert_available",
        ])
    regime_cols = [
        "benchmark_momentum_20d", "benchmark_momentum_60d", "benchmark_vol_20d_ann", "benchmark_vol_60d_ann",
        "benchmark_drawdown_60d", "market_breadth_20d", "cross_sectional_dispersion_20d",
        "avg_universe_drawdown_60d", "pct_above_sma50",
    ]
    for expert in EXPERTS:
        for regime_col in ["benchmark_vol_20d_ann", "benchmark_drawdown_60d", "cross_sectional_dispersion_20d", "market_breadth_20d"]:
            col = f"{expert}_score_x_{regime_col}"
            frame[col] = frame[f"{expert}_expert_score"] * frame[regime_col]
            base_feature_cols.append(col)
    feature_cols = base_feature_cols + regime_cols
    frame[feature_cols] = frame[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return frame.sort_values(["date", "ticker"]).reset_index(drop=True), feature_cols, regime


## 4. Walk-Forward Ridge Router

The router retrains only on rows whose forward target was already available before the rebalance date. That is the notebook's main leakage guard.


In [ ]:
def _coerce_datetime_columns(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    out = frame.copy()
    for col in columns:
        out[col] = pd.to_datetime(out[col], utc=True).dt.tz_localize(None)
    return out


def _fit_meta_model(train_frame: pd.DataFrame, feature_cols: list[str]) -> Pipeline:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=META_ALPHA)),
    ])
    X = train_frame[feature_cols].astype(float).replace([np.inf, -np.inf], 0.0).fillna(0.0)
    y = train_frame["target_rank_alpha_5d"].astype(float)
    model.fit(X, y)
    return model


def _expert_contribution_columns(model: Pipeline, rows: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    scaler = model.named_steps["scaler"]
    ridge = model.named_steps["ridge"]
    X = rows[feature_cols].astype(float).replace([np.inf, -np.inf], 0.0).fillna(0.0)
    X_scaled = scaler.transform(X)
    raw_contrib = pd.DataFrame(X_scaled * ridge.coef_, index=rows.index, columns=feature_cols)
    out = pd.DataFrame(index=rows.index)
    for expert in EXPERTS:
        expert_cols = [col for col in raw_contrib.columns if col.startswith(f"{expert}_")]
        out[f"contribution_{expert}"] = raw_contrib[expert_cols].sum(axis=1) if expert_cols else 0.0
    out["contribution_regime"] = raw_contrib[[col for col in raw_contrib.columns if not any(col.startswith(f"{expert}_") for expert in EXPERTS)]].sum(axis=1)
    return out


def _walk_forward_meta_predictions(meta_frame: pd.DataFrame, feature_cols: list[str], eval_dates: pd.DatetimeIndex) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    frame = _coerce_datetime_columns(meta_frame, ["date", "target_available_date"])
    eval_dates = pd.DatetimeIndex(pd.to_datetime(eval_dates)).sort_values()
    predictions = []
    coefficients = []
    train_audit = []
    current_model = None
    current_model_date = None

    for i, date_value in enumerate(eval_dates):
        known = frame.loc[
            (frame["date"] < date_value)
            & (frame["target_available_date"].notna())
            & (frame["target_available_date"] < date_value)
            & (frame["target_rank_alpha_5d"].notna())
        ].copy()
        train_date_count = int(known["date"].nunique())
        should_retrain = current_model is None or i % META_RETRAIN_EVERY_N_REBALANCES == 0
        if train_date_count < META_MIN_TRAIN_DATES:
            today = frame.loc[frame["date"] == date_value].copy()
            if not today.empty:
                score_cols = [f"{expert}_expert_score" for expert in EXPERTS if f"{expert}_expert_score" in today.columns]
                today["expected_return"] = today[score_cols].mean(axis=1) if score_cols else 0.0
                today["horizon"] = META_TARGET_HORIZON
                today["meta_model_train_date"] = pd.NaT
                for expert in EXPERTS:
                    score_col = f"{expert}_expert_score"
                    today[f"contribution_{expert}"] = today[score_col] / max(len(EXPERTS), 1) if score_col in today.columns else 0.0
                today["contribution_regime"] = 0.0
                predictions.append(today)
            train_audit.append({
                "date": date_value,
                "train_rows": int(len(known)),
                "train_dates": train_date_count,
                "max_target_available_date": known["target_available_date"].max() if not known.empty else pd.NaT,
                "trained": False,
                "reason": "cold_start_equal_expert_blend",
            })
            continue
        if should_retrain:
            current_model = _fit_meta_model(known, feature_cols)
            current_model_date = date_value
            ridge = current_model.named_steps["ridge"]
            coefficients.append(pd.DataFrame({
                "date": date_value,
                "feature": feature_cols,
                "coefficient_scaled_space": ridge.coef_,
                "intercept": ridge.intercept_,
                "train_rows": len(known),
                "train_dates": train_date_count,
            }))
        today = frame.loc[frame["date"] == date_value].copy()
        if today.empty:
            train_audit.append({
                "date": date_value,
                "train_rows": int(len(known)),
                "train_dates": train_date_count,
                "max_target_available_date": known["target_available_date"].max() if not known.empty else pd.NaT,
                "trained": bool(should_retrain),
                "reason": "no_today_rows",
            })
            continue
        X_today = today[feature_cols].astype(float).replace([np.inf, -np.inf], 0.0).fillna(0.0)
        today["expected_return"] = current_model.predict(X_today)
        today["horizon"] = META_TARGET_HORIZON
        today["meta_model_train_date"] = current_model_date
        contrib = _expert_contribution_columns(current_model, today, feature_cols)
        today = pd.concat([today.reset_index(drop=True), contrib.reset_index(drop=True)], axis=1)
        predictions.append(today)
        train_audit.append({
            "date": date_value,
            "train_rows": int(len(known)),
            "train_dates": train_date_count,
            "max_target_available_date": known["target_available_date"].max() if not known.empty else pd.NaT,
            "trained": bool(should_retrain),
            "reason": "ok",
        })

    pred_frame = pd.concat(predictions, ignore_index=True) if predictions else pd.DataFrame()
    coef_frame = pd.concat(coefficients, ignore_index=True) if coefficients else pd.DataFrame(columns=["date", "feature", "coefficient_scaled_space", "intercept", "train_rows", "train_dates"])
    audit_frame = pd.DataFrame(train_audit)
    return pred_frame, coef_frame, audit_frame


## 5. Ensemble Portfolio Builder

This builder turns the router's score into a long-only portfolio with top-name selection, a 10% cap, and turnover smoothing.


In [ ]:
def _project_capped_simplex(weights: pd.Series, cap: float) -> pd.Series:
    w = weights.clip(lower=0.0).astype(float).copy()
    if w.sum() <= 0:
        return w
    w = w / w.sum()
    active_count = int((w > 0).sum())
    effective_cap = max(float(cap), 1.0 / max(active_count, 1))
    for _ in range(100):
        over = w > effective_cap
        if not over.any():
            break
        excess = float((w.loc[over] - effective_cap).sum())
        w.loc[over] = effective_cap
        under = (w > 0) & (w < effective_cap - 1e-12)
        if not under.any():
            break
        base = w.loc[under]
        if base.sum() <= 0:
            w.loc[under] += excess / int(under.sum())
        else:
            w.loc[under] += excess * base / base.sum()
    total = float(w.sum())
    return w / total if total > 0 else w


def _build_ensemble_weights(predictions: pd.DataFrame, spec) -> PortfolioWeights:
    if predictions.empty:
        raise ValueError("No ensemble predictions were produced. Check meta training history and expert artifacts.")
    benchmark = spec.benchmark_ticker.upper()
    tradable = [ticker.upper() for ticker in spec.tickers if ticker.upper() != benchmark]
    dates = pd.DatetimeIndex(sorted(pd.to_datetime(predictions["date"]).unique()))
    weights = pd.DataFrame(0.0, index=dates, columns=tradable)
    previous = pd.Series(0.0, index=tradable)

    for date_value in dates:
        frame = predictions.loc[predictions["date"] == date_value].copy()
        frame["ticker"] = frame["ticker"].astype(str).str.upper()
        frame = frame.loc[frame["ticker"].isin(tradable)]
        if frame.empty:
            continue
        universe_size = int(frame["ticker"].nunique())
        k = min(30, max(12, int(0.40 * universe_size)))
        chosen = frame.sort_values("expected_return", ascending=False).head(k).reset_index(drop=True)
        ranks = pd.Series(np.arange(len(chosen), 0, -1, dtype=float), index=chosen["ticker"])
        raw = pd.Series(0.0, index=tradable)
        raw.loc[ranks.index] = ranks / ranks.sum()
        capped = _project_capped_simplex(raw, MAX_WEIGHT_CAP).reindex(tradable).fillna(0.0)
        smoothed = SMOOTH_PREVIOUS_WEIGHT * previous + SMOOTH_NEW_WEIGHT * capped
        if smoothed.sum() <= 0:
            smoothed = capped
        smoothed = smoothed / smoothed.sum()
        weights.loc[date_value] = smoothed
        previous = smoothed

    weights.index.name = "date"
    weights = validate_weights_frame(weights, dataset_name=spec, repo_root=repo_root)
    return PortfolioWeights(
        weights=weights,
        dataset_name=spec.identifier,
        strategy_name=ENSEMBLE_NAME,
        metadata={
            "router_type": "trained_walk_forward_ridge_soft_blend",
            "rebalance_frequency": REBALANCE_FREQUENCY,
            "max_weight_cap": MAX_WEIGHT_CAP,
            "smooth_previous_weight": SMOOTH_PREVIOUS_WEIGHT,
            "smooth_new_weight": SMOOTH_NEW_WEIGHT,
        },
    )


def _expert_contribution_by_date(predictions: pd.DataFrame) -> pd.DataFrame:
    if predictions.empty:
        return pd.DataFrame()
    contrib_cols = [f"contribution_{expert}" for expert in EXPERTS]
    available_cols = [col for col in contrib_cols if col in predictions.columns]
    grouped = predictions.groupby("date", sort=True)[available_cols].mean().reset_index()
    if available_cols:
        grouped["top_contributor"] = grouped[available_cols].abs().idxmax(axis=1).str.replace("contribution_", "", regex=False)
    return grouped


## 6. Run One Dataset

This is the main reusable entry point. Change `ENSEMBLE_DATASETS` above and rerun from here.


In [ ]:
def mlflow_tracking_preflight(tracking_uri: str, timeout: float = 5.0) -> tuple[bool, str]:
    parsed = urlparse(tracking_uri)
    if parsed.scheme not in {"http", "https"}:
        return True, "local tracking URI"
    if not parsed.hostname:
        return False, f"remote tracking URI has no hostname: {tracking_uri}"
    port = parsed.port or (443 if parsed.scheme == "https" else 80)
    try:
        with socket.create_connection((parsed.hostname, port), timeout=timeout):
            return True, f"{parsed.hostname}:{port} reachable"
    except OSError as exc:
        return False, f"{parsed.hostname}:{port} not reachable ({exc})"


def _write_dataframe(path: Path, frame: pd.DataFrame, *, index: bool = False) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_parquet(path, index=index)
    return str(path)


def _log_artifact_if_exists(path: str | Path, artifact_path: str | None = None) -> None:
    p = Path(path)
    if p.exists():
        mlflow.log_artifact(str(p), artifact_path=artifact_path)


def _assert_no_target_leakage(train_audit: pd.DataFrame, meta_frame: pd.DataFrame) -> None:
    if train_audit.empty or "max_target_available_date" not in train_audit.columns:
        return
    audit = _coerce_datetime_columns(train_audit, ["date", "max_target_available_date"])
    bad = audit.loc[
        audit["max_target_available_date"].notna()
        & (audit["max_target_available_date"] >= audit["date"])
    ]
    if not bad.empty:
        preview = bad[["date", "max_target_available_date", "train_rows", "train_dates"]].head().to_dict("records")
        raise AssertionError(f"Target leakage detected in meta training windows: {preview}")


def run_ensemble_dataset(dataset_name: str) -> dict[str, object]:
    spec = get_dataset_spec(dataset_name, repo_root=repo_root)
    benchmark = spec.benchmark_ticker.upper()
    tradable = [ticker.upper() for ticker in spec.tickers if ticker.upper() != benchmark]
    prices = load_prices(spec, repo_root=repo_root)
    price_wide = _pivot_adj_close(prices)
    test_dates = pd.DatetimeIndex(price_wide.loc[str(spec.test_start): str(spec.test_end)].index)
    rebalance_dates = _select_weekly_first_trading_dates(test_dates)
    if FAST_SMOKE:
        rebalance_dates = rebalance_dates[:80]
    if rebalance_dates.empty:
        raise ValueError(f"No rebalance dates found for {dataset_name}")

    expert_runs = []
    for expert in EXPERTS:
        expert_runs.append(run_expert_on_dataset(expert, dataset_name, rebalance_dates, tradable))
    expert_signals = pd.concat([run.expert_signal for run in expert_runs], ignore_index=True)
    meta_frame, feature_cols, regime_features = _build_meta_frame(dataset_name, expert_signals, prices, spec)
    predictions, coefficients, train_audit = _walk_forward_meta_predictions(meta_frame, feature_cols, rebalance_dates)
    if predictions.empty:
        raise ValueError(
            f"No ensemble predictions for {dataset_name}. Try lowering META_MIN_TRAIN_DATES for research or ensure expert artifacts cover pre-test history."
        )
    portfolio = _build_ensemble_weights(predictions, spec)
    if benchmark in portfolio.weights.columns:
        raise AssertionError("Benchmark ticker leaked into ensemble tradable weights")
    row_sums = portfolio.weights.sum(axis=1)
    if not np.allclose(row_sums.values, 1.0, atol=1e-8):
        raise AssertionError("Ensemble weight rows do not sum to 1.0")
    _assert_no_target_leakage(train_audit, meta_frame)

    result = backtest_weights(spec, portfolio, benchmark=benchmark, repo_root=repo_root)
    output_dir = OUTPUT_ROOT / dataset_name
    if FAST_SMOKE:
        output_dir = OUTPUT_ROOT / f"_smoke_{dataset_name}"
    output_dir.mkdir(parents=True, exist_ok=True)

    artifact_paths = {}
    artifact_paths["expert_signals"] = _write_dataframe(output_dir / "expert_signals.parquet", expert_signals)
    artifact_paths["meta_training_frame"] = _write_dataframe(output_dir / "meta_training_frame.parquet", meta_frame)
    artifact_paths["ensemble_predictions"] = _write_dataframe(output_dir / "ensemble_predictions.parquet", predictions)
    artifact_paths["ensemble_weights"] = _write_dataframe(output_dir / "ensemble_weights.parquet", portfolio.weights, index=True)
    contribution_by_date = _expert_contribution_by_date(predictions)
    artifact_paths["expert_contribution_by_date"] = _write_dataframe(output_dir / "expert_contribution_by_date.parquet", contribution_by_date)
    artifact_paths["meta_model_coefficients"] = _write_dataframe(output_dir / "meta_model_coefficients.parquet", coefficients)
    artifact_paths["train_audit"] = _write_dataframe(output_dir / "walk_forward_train_audit.parquet", train_audit)

    config = {
        "ensemble_name": ENSEMBLE_NAME,
        "dataset_name": dataset_name,
        "benchmark_ticker": benchmark,
        "experts": {run.expert: run.metadata for run in expert_runs},
        "target_horizon": META_TARGET_HORIZON,
        "meta_model": f"Ridge(alpha={META_ALPHA})",
        "min_train_dates": META_MIN_TRAIN_DATES,
        "retrain_every_n_rebalances": META_RETRAIN_EVERY_N_REBALANCES,
        "rebalance_frequency": REBALANCE_FREQUENCY,
        "max_weight_cap": MAX_WEIGHT_CAP,
        "smooth_previous_weight": SMOOTH_PREVIOUS_WEIGHT,
        "smooth_new_weight": SMOOTH_NEW_WEIGHT,
        "feature_columns": feature_cols,
    }
    config_path = output_dir / "ensemble_config.json"
    config_path.write_text(json.dumps(config, indent=2, sort_keys=True, default=str) + "\n", encoding="utf-8")
    artifact_paths["ensemble_config"] = str(config_path)

    backtest_artifacts = write_backtest_artifacts(result, output_dir / "backtest")
    artifact_paths.update({f"backtest_{key}": value for key, value in backtest_artifacts.items()})
    result.artifact_paths.update(backtest_artifacts)

    active_cols = [f"{expert}_expert_active" for expert in EXPERTS]
    available_cols = [f"{expert}_expert_available" for expert in EXPERTS]
    extra_metrics = {
        "average_active_expert_count": float((meta_frame[active_cols] > 0).sum(axis=1).mean()),
        "average_available_expert_count": float((meta_frame[available_cols] > 0).sum(axis=1).mean()),
        "meta_prediction_rows": float(len(predictions)),
        "meta_train_rebalance_count": float((train_audit["reason"] == "ok").sum()),
    }
    if not contribution_by_date.empty and "top_contributor" in contribution_by_date.columns:
        shares = contribution_by_date["top_contributor"].value_counts(normalize=True)
        for expert in EXPERTS:
            extra_metrics[f"top_contributor_share_{expert}"] = float(shares.get(expert, 0.0))
            col = f"contribution_{expert}"
            if col in contribution_by_date.columns:
                extra_metrics[f"avg_contribution_{expert}"] = float(contribution_by_date[col].mean())
    result.metrics.update(extra_metrics)

    return {
        "dataset_name": dataset_name,
        "spec": spec,
        "prices": prices,
        "expert_runs": expert_runs,
        "expert_signals": expert_signals,
        "meta_frame": meta_frame,
        "predictions": predictions,
        "coefficients": coefficients,
        "train_audit": train_audit,
        "portfolio": portfolio,
        "result": result,
        "artifact_paths": artifact_paths,
        "config": config,
    }


## 7. Execute Ensemble Backtests

This cell runs every configured dataset. MLflow logging is disabled by default so smoke tests do not require the tracking server. Set `PUTAH_ENSEMBLE_LOG_TO_MLFLOW=1` to log.


In [ ]:
ensemble_runs = {}
summary_rows = []
for dataset_name in ENSEMBLE_DATASETS:
    print(f"Running Putah Creek meta-ensemble: {dataset_name}", flush=True)
    payload = run_ensemble_dataset(dataset_name)
    ensemble_runs[dataset_name] = payload
    metrics = dict(payload["result"].metrics)
    summary_rows.append({
        "dataset_name": dataset_name,
        "benchmark_ticker": payload["spec"].benchmark_ticker,
        "test_start": payload["spec"].test_start,
        "test_end": payload["spec"].test_end,
        "total_return": metrics.get("total_return"),
        "annual_return": metrics.get("annual_return"),
        "annual_volatility": metrics.get("annual_volatility"),
        "sharpe": metrics.get("sharpe"),
        "sortino": metrics.get("sortino"),
        "max_drawdown": metrics.get("max_drawdown"),
        "benchmark_total_return": metrics.get("benchmark_total_return"),
        "excess_return_vs_benchmark": metrics.get("excess_return_vs_benchmark"),
        "average_turnover": metrics.get("average_turnover"),
        "prediction_rows": len(payload["predictions"]),
        "weight_rows": len(payload["portfolio"].weights),
    })

ensemble_summary = pd.DataFrame(summary_rows)
summary_path = OUTPUT_ROOT / ("_smoke_ensemble_summary.parquet" if FAST_SMOKE else "ensemble_summary.parquet")
summary_csv_path = summary_path.with_suffix(".csv")
ensemble_summary.to_parquet(summary_path, index=False)
ensemble_summary.to_csv(summary_csv_path, index=False)
display(ensemble_summary)
print("Summary:", summary_path)


## 8. MLflow Logging

Logs one ensemble run per dataset with standard backtest artifacts, predictions, weights, coefficients, expert contribution diagnostics, and the ensemble config.


In [ ]:
if LOG_TO_MLFLOW:
    layout = init_mlflow(repo_root=repo_root)
    ok, reason = mlflow_tracking_preflight(layout["tracking_uri"])
    if not ok:
        message = f"MLflow tracking preflight failed: {reason}"
        if MLFLOW_REQUIRED:
            raise ConnectionError(message)
        print("MLflow logging skipped:", message)
    else:
        for dataset_name, payload in ensemble_runs.items():
            spec = payload["spec"]
            result = payload["result"]
            portfolio = payload["portfolio"]
            predictions = payload["predictions"]
            with start_run(
                run_name=f"{ENSEMBLE_NAME}_{dataset_name}",
                dataset_name=spec,
                repo_root=repo_root,
                tags={
                    "model_family": "ensemble",
                    "router_type": "trained_walk_forward_ridge_soft_blend",
                    "base_models": ",".join(EXPERTS),
                    "rebalance_frequency": REBALANCE_FREQUENCY,
                    "workflow": "putah_creek_meta_ensemble_runner",
                },
            ):
                mlflow.log_params({
                    "model_name": ENSEMBLE_NAME,
                    "dataset_name": dataset_name,
                    "target_horizon": META_TARGET_HORIZON,
                    "meta_model": f"Ridge(alpha={META_ALPHA})",
                    "min_train_dates": META_MIN_TRAIN_DATES,
                    "retrain_every_n_rebalances": META_RETRAIN_EVERY_N_REBALANCES,
                    "max_weight_cap": MAX_WEIGHT_CAP,
                    "smooth_previous_weight": SMOOTH_PREVIOUS_WEIGHT,
                    "smooth_new_weight": SMOOTH_NEW_WEIGHT,
                    "expert_count": len(EXPERTS),
                    "expert_names": ",".join(EXPERTS),
                })
                log_predictions(predictions[["date", "ticker", "horizon", "expected_return"]].copy())
                log_portfolio(portfolio)
                log_backtest(result)
                for key, artifact_path in payload["artifact_paths"].items():
                    _log_artifact_if_exists(artifact_path, artifact_path="putah_creek_meta_ensemble")
        print("MLflow logging complete.")
else:
    print("MLflow logging skipped. Set PUTAH_ENSEMBLE_LOG_TO_MLFLOW=1 to enable it.")


## 9. Acceptance Checks


In [ ]:
for dataset_name, payload in ensemble_runs.items():
    spec = payload["spec"]
    benchmark = spec.benchmark_ticker.upper()
    weights = payload["portfolio"].weights
    assert benchmark not in weights.columns, f"{dataset_name}: benchmark leaked into weights"
    assert np.allclose(weights.sum(axis=1).values, 1.0, atol=1e-8), f"{dataset_name}: weights do not sum to 1"
    assert not payload["predictions"].empty, f"{dataset_name}: empty predictions"
    assert not payload["expert_signals"].empty, f"{dataset_name}: empty expert signals"
    assert Path(payload["artifact_paths"]["ensemble_config"]).exists(), f"{dataset_name}: missing config artifact"
    _assert_no_target_leakage(payload["train_audit"], payload["meta_frame"])
print("All Putah Creek meta-ensemble acceptance checks passed.")
